# Notebook 04: MILP Optimization - Budget Allocation

**PHIÊN BẢN ĐÃ SỬA - Channel Base Cost**

**Các thay đổi chính:**
- ✅ Sử dụng channel_base_cost (không phải spend_usd)
- ✅ Expected value từ prior_spend_180d (không phải revenue_usd)
- ✅ MILP vs Greedy comparison qua 5 periods
- ✅ NO DATA LEAKAGE - chỉ dùng pre-treatment info

**Ngày:** 06/11/2026

**Tác giả:** Phan Hồ Ngọc Hân

In [1]:
# ============================================
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ============================================
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import time

# Optimization
from scipy.optimize import milp, LinearConstraint, Bounds

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"✅ Import hoàn tất - SEED = {SEED}")

✅ Import hoàn tất - SEED = 42


## 1. Load CATE Estimates Từ Notebook 03

In [2]:
# ============================================
# LOAD CATE ESTIMATES TỪ NOTEBOOK 03
# ============================================
data_path = Path("../data/processed/cate_estimates.parquet")
df_cate = pd.read_parquet(data_path)

print(f"✅ Đã load CATE estimates: {len(df_cate):,} customers")
print(f"\n📊 Tóm tắt CATE:")
print(f"   Mean: {df_cate['cate'].mean():+.4f} ({df_cate['cate'].mean()*100:+.2f} pp)")
print(f"   Std:  {df_cate['cate'].std():.4f}")
print(f"   Range: [{df_cate['cate'].min():+.4f}, {df_cate['cate'].max():+.4f}]")

print(f"\n👥 Phân bố 4 Nhóm:")
print(df_cate['group'].value_counts().sort_index())

✅ Đã load CATE estimates: 5,000 customers

📊 Tóm tắt CATE:
   Mean: +0.0565 (+5.65 pp)
   Std:  0.1090
   Range: [-0.4764, +0.6305]

👥 Phân bố 4 Nhóm:
group
Lost Causes      2148
Persuadables     2214
Sleeping Dogs     511
Sure Things       127
Name: count, dtype: int64


## 2. Load Dữ Liệu Gốc Để Lấy Channel Base Cost

In [3]:
# ============================================
# LOAD PROCESSED DATA ĐỂ LẤY CHANNEL_BASE_COST
# ============================================
df_full = pd.read_parquet("../data/processed/marketing_campaign_processed.parquet")

# Merge CATE với original data
df = df_full.merge(df_cate[['user_id', 'cate', 'mu_0', 'mu_1', 'group']], on='user_id', how='left')

print(f"✅ Đã merge dữ liệu: {len(df):,} customers")
print(f"\n📋 Các cột có sẵn:")
print(df.columns.tolist())

✅ Đã merge dữ liệu: 5,024 customers

📋 Các cột có sẵn:
['user_id', 'campaign_id', 'channel', 'device', 'country', 'segment', 'prior_visits_30d', 'prior_spend_180d', 'treatment_exposed', 'impressions', 'clicks', 'spend_usd', 'conversion', 'revenue_usd', 'roi', 'value_tier', 'channel_base_cost', 'temporal_proxy', 'split', 'decision_period', 'cate', 'mu_0', 'mu_1', 'group']


## 3. Định Nghĩa Bài Toán Optimization

**Objective:**
```
Maximize: Σᵢ (CATE_i × expected_value_i × x_i)
```

**Ràng buộc:**
1. **Ngân sách:** Σᵢ (channel_base_cost_i × x_i) ≤ B
2. **Số lượng tối đa:** Σᵢ x_i ≤ C
3. **Binary:** x_i ∈ {0, 1}

**Key Changes:**
- ❌ **KHÔNG** dùng `spend_usd` (post-treatment!)
- ✅ Dùng `channel_base_cost` (pre-determined)
- ❌ **KHÔNG** dùng `revenue_usd` (post-treatment!)
- ✅ Dùng `prior_spend_180d` làm proxy cho expected_value

In [4]:
# ============================================
# CHUẨN BỊ DATA CHO OPTIMIZATION
# ============================================

# Chỉ optimize trên test set
df_test = df[df['split'] == 'locked_test'].copy()

# Expected value = prior_spend_180d (proxy cho customer lifetime value)
df_test['expected_value'] = df_test['prior_spend_180d']

# Objective coefficient = CATE × expected_value
df_test['objective_coef'] = df_test['cate'] * df_test['expected_value']

# Cost = channel_base_cost (pre-determined, NO LEAKAGE)
df_test['cost'] = df_test['channel_base_cost']

print("✅ Đã chuẩn bị dữ liệu optimization")
print(f"\n📊 Test Set: {len(df_test):,} customers")
print(f"\n💰 Thống kê Chi phí (channel_base_cost):")
print(f"   Mean:   ${df_test['cost'].mean():.2f}")
print(f"   Median: ${df_test['cost'].median():.2f}")
print(f"   Range:  [${df_test['cost'].min():.2f}, ${df_test['cost'].max():.2f}]")

print(f"\n📈 Giá trị Kỳ vọng (prior_spend_180d):")
print(f"   Mean:   ${df_test['expected_value'].mean():.2f}")
print(f"   Median: ${df_test['expected_value'].median():.2f}")

print(f"\n🎯 Hệ số Mục tiêu (CATE × expected_value):")
print(f"   Mean:   ${df_test['objective_coef'].mean():.2f}")
print(f"   Median: ${df_test['objective_coef'].median():.2f}")
print(f"   Range:  [${df_test['objective_coef'].min():.2f}, ${df_test['objective_coef'].max():.2f}]")

✅ Đã chuẩn bị dữ liệu optimization

📊 Test Set: 502 customers

💰 Thống kê Chi phí (channel_base_cost):
   Mean:   $2.25
   Median: $2.50
   Range:  [$0.50, $4.00]

📈 Giá trị Kỳ vọng (prior_spend_180d):
   Mean:   $119.38
   Median: $94.75

🎯 Hệ số Mục tiêu (CATE × expected_value):
   Mean:   $10.37
   Median: $3.16
   Range:  [$-66.21, $243.82]


## 4. Thuật Toán Greedy (Baseline)

In [5]:
# ============================================
# GREEDY ALGORITHM
# ============================================

def greedy_selection(df, budget, capacity):
    """
    Greedy algorithm: Sắp xếp theo objective/cost ratio, chọn top customers
    
    Args:
        df: DataFrame với 'objective_coef' và 'cost'
        budget: Ngân sách tối đa
        capacity: Số lượng customers tối đa
    
    Returns:
        selected_ids: List user_ids được chọn
        total_value: Tổng objective value
        total_cost: Tổng cost
        runtime: Thời gian chạy (seconds)
    """
    start_time = time.time()
    
    # Calculate efficiency score
    df_work = df.copy()
    df_work['efficiency'] = df_work['objective_coef'] / df_work['cost']
    
    # Sort by efficiency descending
    df_work = df_work.sort_values('efficiency', ascending=False)
    
    # Greedy selection
    selected = []
    total_cost = 0
    total_value = 0
    
    for idx, row in df_work.iterrows():
        # Check constraints
        if len(selected) >= capacity:
            break
        if total_cost + row['cost'] > budget:
            break
        
        # Add customer
        selected.append(row['user_id'])
        total_cost += row['cost']
        total_value += row['objective_coef']
    
    runtime = time.time() - start_time
    
    return selected, total_value, total_cost, runtime

print("✅ Đã định nghĩa thuật toán Greedy")

✅ Đã định nghĩa thuật toán Greedy


## 5. MILP Solver (Exact Optimization)

In [6]:
# ============================================
# MILP SOLVER
# ============================================

def milp_selection(df, budget, capacity):
    """
    MILP optimization: Solve exact binary knapsack problem
    
    Args:
        df: DataFrame với 'objective_coef' và 'cost'
        budget: Ngân sách tối đa
        capacity: Số lượng customers tối đa
    
    Returns:
        selected_ids: List user_ids được chọn
        total_value: Tổng objective value
        total_cost: Tổng cost
        runtime: Thời gian chạy (seconds)
    """
    start_time = time.time()
    
    n = len(df)
    
    # Objective: maximize Σ(objective_coef × x)
    c = -df['objective_coef'].values  # Negative vì scipy minimizes
    
    # Ràng buộc:
    # 1. Ngân sách: Σ(cost × x) <= budget
    # 2. Số lượng tối đa: Σ(x) <= capacity
    A = np.vstack([
        df['cost'].values,     # Budget constraint
        np.ones(n)             # Capacity constraint
    ])
    
    b_u = np.array([budget, capacity])
    b_l = np.array([-np.inf, -np.inf])
    
    constraints = LinearConstraint(A, b_l, b_u)
    
    # Binary variables
    integrality = np.ones(n)  # 1 = integer constraint
    bounds = Bounds(lb=0, ub=1)  # Binary: 0 or 1
    
    # Solve
    result = milp(
        c=c,
        constraints=constraints,
        integrality=integrality,
        bounds=bounds
    )
    
    runtime = time.time() - start_time
    
    # Extract solution
    if result.success:
        x_opt = result.x
        selected_mask = (x_opt > 0.5)  # Binary: round to 0 or 1
        selected_ids = df.loc[selected_mask, 'user_id'].tolist()
        total_value = -result.fun  # Negate back (was minimizing -objective)
        total_cost = (df.loc[selected_mask, 'cost'].sum())
        
        return selected_ids, total_value, total_cost, runtime
    else:
        print(f"⚠️  MILP failed: {result.message}")
        return [], 0, 0, runtime

print("✅ Đã định nghĩa MILP solver")

✅ Đã định nghĩa MILP solver


## 6. Test Optimization - Single Period

In [7]:
# ============================================
# TEST SINGLE PERIOD
# ============================================

BUDGET = 10_000  # $10K budget
CAPACITY = 200   # 200 customers max

print("=" * 70)
print("TEST OPTIMIZATION - MỘT KỲ")
print("=" * 70)

print(f"\n📋 Ràng buộc:")
print(f"   Ngân sách:   ${BUDGET:,}")
print(f"   Số lượng tối đa: {CAPACITY:,} customers")

# Greedy
print(f"\n⏳ Đang chạy Greedy...")
greedy_ids, greedy_value, greedy_cost, greedy_time = greedy_selection(
    df_test, BUDGET, CAPACITY
)

print(f"\n✅ Kết quả Greedy:")
print(f"   Đã chọn:  {len(greedy_ids):,} customers")
print(f"   Giá trị:     ${greedy_value:,.2f}")
print(f"   Chi phí:      ${greedy_cost:,.2f}")
print(f"   Thời gian:   {greedy_time*1000:.1f} ms")

# MILP
print(f"\n⏳ Đang chạy MILP...")
milp_ids, milp_value, milp_cost, milp_time = milp_selection(
    df_test, BUDGET, CAPACITY
)

print(f"\n✅ Kết quả MILP:")
print(f"   Đã chọn:  {len(milp_ids):,} customers")
print(f"   Giá trị:     ${milp_value:,.2f}")
print(f"   Chi phí:      ${milp_cost:,.2f}")
print(f"   Thời gian:   {milp_time*1000:.1f} ms")

# Comparison
value_gain = milp_value - greedy_value
pct_gain = (value_gain / greedy_value * 100) if greedy_value > 0 else 0
speedup = milp_time / greedy_time if greedy_time > 0 else 0

print(f"\n📊 So sánh:")
print(f"   MILP tốt hơn:     ${value_gain:,.2f} ({pct_gain:+.2f}%)")
print(f"   MILP Slowdown: {speedup:.1f}×")

if abs(pct_gain) < 0.01:
    print(f"\n⚠️  MILP và Greedy cho kết quả giống nhau!")
    print(f"   → Greedy tối ưu với cấu trúc ràng buộc này")
    print(f"   → Nên dùng Greedy (faster {speedup:.1f}×)")
else:
    print(f"\n✅ MILP tốt hơn Greedy {pct_gain:.2f}%")
    print(f"   → Worth the {speedup:.1f}× slower runtime")

TEST OPTIMIZATION - MỘT KỲ

📋 Ràng buộc:
   Ngân sách:   $10,000
   Số lượng tối đa: 200 customers

⏳ Đang chạy Greedy...

✅ Kết quả Greedy:
   Đã chọn:  200 customers
   Giá trị:     $5,423.51
   Chi phí:      $367.70
   Thời gian:   3.6 ms

⏳ Đang chạy MILP...

✅ Kết quả MILP:
   Đã chọn:  200 customers
   Giá trị:     $5,532.40
   Chi phí:      $431.60
   Thời gian:   28.8 ms

📊 So sánh:
   MILP tốt hơn:     $108.89 (+2.01%)
   MILP Slowdown: 8.1×

✅ MILP tốt hơn Greedy 2.01%
   → Worth the 8.1× slower runtime


## 7. Multi-Period Simulation (5 Periods)

In [8]:
# ============================================
# 5-PERIOD SIMULATION
# ============================================

NUM_PERIODS = 5
PERIOD_BUDGET = 10_000
PERIOD_CAPACITY = 200

print("=" * 70)
print(f"SIMULATION ĐA KỲ - {NUM_PERIODS} PERIODS")
print("=" * 70)

print(f"\n📋 Ràng buộc mỗi period:")
print(f"   Ngân sách:   ${PERIOD_BUDGET:,}")
print(f"   Số lượng tối đa: {PERIOD_CAPACITY:,} customers")

# Simulate periods với resampling
results = []

for period in range(1, NUM_PERIODS + 1):
    print(f"\n⏳ Period {period}...")
    
    # Resample test set (simulate different customer pools)
    df_period = df_test.sample(n=len(df_test), replace=True, random_state=SEED + period)
    
    # Greedy
    g_ids, g_val, g_cost, g_time = greedy_selection(
        df_period, PERIOD_BUDGET, PERIOD_CAPACITY
    )
    
    # MILP
    m_ids, m_val, m_cost, m_time = milp_selection(
        df_period, PERIOD_BUDGET, PERIOD_CAPACITY
    )
    
    # Store results
    results.append({
        'period': period,
        'greedy_value': g_val,
        'greedy_n': len(g_ids),
        'greedy_cost': g_cost,
        'greedy_time': g_time,
        'milp_value': m_val,
        'milp_n': len(m_ids),
        'milp_cost': m_cost,
        'milp_time': m_time,
        'gain': m_val - g_val,
        'gain_pct': ((m_val - g_val) / g_val * 100) if g_val > 0 else 0
    })

# Convert to DataFrame
df_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("KẾT QUẢ 5 KỲ")
print("=" * 70)

print("\n📊 Per-Period Results:")
print(df_results[['period', 'greedy_value', 'greedy_n', 'milp_value', 'milp_n', 'gain', 'gain_pct']].to_string(index=False))

# Summary statistics
print("\n📈 Thống kê Tổng hợp:")
print(f"   Avg Greedy Value:  ${df_results['greedy_value'].mean():,.2f}")
print(f"   Avg MILP Value:    ${df_results['milp_value'].mean():,.2f}")
print(f"   Avg Gain:          ${df_results['gain'].mean():,.2f} ({df_results['gain_pct'].mean():+.2f}%)")
print(f"   Avg Greedy Time:   {df_results['greedy_time'].mean()*1000:.1f} ms")
print(f"   Avg MILP Time:     {df_results['milp_time'].mean()*1000:.1f} ms")
print(f"   Greedy Speedup:  {(df_results['greedy_time'].mean() / df_results['milp_time'].mean()):.1f}×")

# Conclusion
avg_gain_pct = df_results['gain_pct'].mean()
if abs(avg_gain_pct) < 0.1:
    print("\n🎯 KẾT LUẬN:")
    print("   ✅ Greedy = MILP (0% gain trung bình)")
    print("   → Cấu trúc ràng buộc đơn giản → Greedy tối ưu")
    print("   → Khuyến nghị: Dùng Greedy (nhanh hơn, kết quả giống nhau)")
else:
    print("\n🎯 KẾT LUẬN:")
    print(f"   ✅ MILP tốt hơn Greedy {avg_gain_pct:.2f}% trung bình")
    print("   → Cost variation lớn → MILP optimizes better")
    print("   → Recommend: Use MILP despite slower runtime")

SIMULATION ĐA KỲ - 5 PERIODS

📋 Ràng buộc mỗi period:
   Ngân sách:   $10,000
   Số lượng tối đa: 200 customers

⏳ Period 1...

⏳ Period 2...



⏳ Period 3...

⏳ Period 4...

⏳ Period 5...

KẾT QUẢ 5 KỲ

📊 Per-Period Results:
 period  greedy_value  greedy_n  milp_value  milp_n     gain  gain_pct
      1     5335.4616       200   5438.2239     200 102.7623    1.9260
      2     5806.7445       200   5946.5325     200 139.7880    2.4073
      3     5852.9776       200   5940.5956     200  87.6180    1.4970
      4     5540.0777       200   5640.0294     200  99.9517    1.8042
      5     4731.9835       200   4833.3757     200 101.3922    2.1427

📈 Thống kê Tổng hợp:
   Avg Greedy Value:  $5,453.45
   Avg MILP Value:    $5,559.75
   Avg Gain:          $106.30 (+1.96%)
   Avg Greedy Time:   2.4 ms
   Avg MILP Time:     13.7 ms
   Greedy Speedup:  0.2×

🎯 KẾT LUẬN:
   ✅ MILP tốt hơn Greedy 1.96% trung bình
   → Cost variation lớn → MILP optimizes better
   → Recommend: Use MILP despite slower runtime


## 8. Phân Tích: Tại Sao MILP Beats Greedy?

In [9]:
# ============================================
# PHÂN TÍCH CONSTRAINT BINDING
# ============================================

print("=" * 70)
print("PHÂN TÍCH: TẠI SAO MILP BEATS GREEDY?")
print("=" * 70)

# Kiểm tra constraint nào binding
print(f"\n🔍 Constraint Analysis (Period 1):")
print(f"\n📊 Giải pháp Greedy:")
print(f"   Customers đã chọn: {len(greedy_ids):,} / {CAPACITY:,} ({len(greedy_ids)/CAPACITY*100:.1f}%)")
print(f"   Ngân sách đã dùng:        ${greedy_cost:,.2f} / ${BUDGET:,} ({greedy_cost/BUDGET*100:.1f}%)")

if len(greedy_ids) == CAPACITY:
    print(f"\n✅ Ràng buộc SỐ LƯỢNG đang BINDING")
    print(f"   → Dừng vì đã chọn đủ {CAPACITY} customers")
else:
    print(f"\n⚠️  Ràng buộc SỐ LƯỢNG KHÔNG binding")

if greedy_cost >= 0.95 * BUDGET:
    print(f"\n✅ Ràng buộc NGÂN SÁCH đang BINDING")
    print(f"   → Dừng vì ngân sách gần hết")
else:
    print(f"\n⚠️  Ràng buộc NGÂN SÁCH KHÔNG binding")
    print(f"   → Ngân sách còn dư: ${BUDGET - greedy_cost:,.2f}")

print(f"\n🎯 Điều kiện Tối ưu của Greedy:")
print(f"   1. Bài toán knapsack đơn (1 ràng buộc ngân sách)")
print(f"   2. Không có ràng buộc liên kết")
print(f"   3. Số lượng binding trước ngân sách")
print(f"\n   → Khi số lượng binding TRƯỚC ngân sách:")
print(f"      Bài toán = 'Chọn top {CAPACITY} theo điểm'")
print(f"      → Greedy TỐI ƯU!")

print(f"\n💡 Khi Nào MILP Thắng?")
print(f"   • Nhiều ràng buộc binding (ngân sách VÀ số lượng đều chặt)")
print(f"   • Giới hạn kênh (max 50 email, min 20 video)")
print(f"   • Yêu cầu segment (≥30% phải là VIP)")
print(f"   • Hiệu ứng tương tác (liên hệ A ảnh hưởng giá trị B)")
print(f"\n   → Dataset hiện tại: Cấu trúc đơn giản → Greedy tối ưu")

print(f"\n💡 Tại Sao MILP Vẫn Thắng (Mặc Dù Capacity Binding)?")
print(f"   1. Greedy sorts by: efficiency = objective/cost")
print(f"   2. Assumes: maximize Σ(obj/cost) ≈ maximize Σ(obj)")
print(f"   3. WRONG khi cost variation lớn!")
print(f"")
print(f"   Example:")
print(f"   • Customer A: obj=$100, cost=$4 → eff=25")
print(f"   • Customer B: obj=$80, cost=$1 → eff=80")
print(f"")
print(f"   Greedy chọn B (higher efficiency)")
print(f"   MILP chọn A (higher absolute value)")
print(f"   → Khi budget dư ($9,600+ còn lại): A is better!")
print(f"")
print(f"   Cost range trong dataset: $0.50 - $4.00 (8× difference!)")
print(f"   → Greedy phạt các kênh đắt tiền quá mức")
print(f"   → MILP tìm được mix tối ưu: value cao hơn ~2%")

PHÂN TÍCH: TẠI SAO MILP BEATS GREEDY?

🔍 Constraint Analysis (Period 1):

📊 Giải pháp Greedy:
   Customers đã chọn: 200 / 200 (100.0%)
   Ngân sách đã dùng:        $367.70 / $10,000 (3.7%)

✅ Ràng buộc SỐ LƯỢNG đang BINDING
   → Dừng vì đã chọn đủ 200 customers

⚠️  Ràng buộc NGÂN SÁCH KHÔNG binding
   → Ngân sách còn dư: $9,632.30

🎯 Điều kiện Tối ưu của Greedy:
   1. Bài toán knapsack đơn (1 ràng buộc ngân sách)
   2. Không có ràng buộc liên kết
   3. Số lượng binding trước ngân sách

   → Khi số lượng binding TRƯỚC ngân sách:
      Bài toán = 'Chọn top 200 theo điểm'
      → Greedy TỐI ƯU!

💡 Khi Nào MILP Thắng?
   • Nhiều ràng buộc binding (ngân sách VÀ số lượng đều chặt)
   • Giới hạn kênh (max 50 email, min 20 video)
   • Yêu cầu segment (≥30% phải là VIP)
   • Hiệu ứng tương tác (liên hệ A ảnh hưởng giá trị B)

   → Dataset hiện tại: Cấu trúc đơn giản → Greedy tối ưu

💡 Tại Sao MILP Vẫn Thắng (Mặc Dù Capacity Binding)?
   1. Greedy sorts by: efficiency = objective/cost
   2. Ass

## 9. Xuất Kết Quả

In [10]:
# ============================================
# XUẤT OPTIMIZATION RESULTS
# ============================================

print("\n" + "=" * 70)
print("XUẤT KẾT QUẢ")
print("=" * 70)

output_dir = Path("../data/processed")

# Save multi-period results
results_path = output_dir / "optimization_results.parquet"
df_results.to_parquet(results_path, index=False)
print(f"\n✅ Đã lưu kết quả: {results_path}")

# Save summary statistics
summary = {
    'num_periods': NUM_PERIODS,
    'budget_per_period': PERIOD_BUDGET,
    'capacity_per_period': PERIOD_CAPACITY,
    'avg_greedy_value': float(df_results['greedy_value'].mean()),
    'avg_milp_value': float(df_results['milp_value'].mean()),
    'avg_gain': float(df_results['gain'].mean()),
    'avg_gain_pct': float(df_results['gain_pct'].mean()),
    'avg_greedy_time_ms': float(df_results['greedy_time'].mean() * 1000),
    'avg_milp_time_ms': float(df_results['milp_time'].mean() * 1000),
    'speedup': float(df_results['milp_time'].mean() / df_results['greedy_time'].mean()),
    'recommendation': 'Greedy' if abs(avg_gain_pct) < 0.1 else 'MILP',
    'reason': 'Simple constraints - Greedy optimal' if abs(avg_gain_pct) < 0.1 else 'Cost variation lớn → MILP superior'
}

summary_path = output_dir / "optimization_summary.json"
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Đã lưu tóm tắt: {summary_path}")

# Save selected customer IDs for Period 1
selection_df = pd.DataFrame({
    'user_id': greedy_ids,
    'method': 'greedy',
    'period': 1
})

selection_path = output_dir / "optimization_selections.parquet"
selection_df.to_parquet(selection_path, index=False)
print(f"✅ Đã lưu danh sách chọn: {selection_path}")
print(f"   {len(selection_df):,} customers selected (Period 1)")


XUẤT KẾT QUẢ

✅ Đã lưu kết quả: ../data/processed/optimization_results.parquet
✅ Đã lưu tóm tắt: ../data/processed/optimization_summary.json
✅ Đã lưu danh sách chọn: ../data/processed/optimization_selections.parquet
   200 customers selected (Period 1)


## 10. Tóm Tắt Cuối Cùng

In [11]:
# ============================================
# TÓM TẮT CUỐI CÙNG
# ============================================

print("\n" + "=" * 70)
print("TÓM TẮT NOTEBOOK 04")
print("=" * 70)

print(f"\n✅ Các Phương Pháp Optimization So Sánh:")
print(f"   • Greedy: Sắp xếp theo hiệu suất, chọn top")
print(f"   • MILP:   Giải chính xác bài toán binary knapsack")

print(f"\n✅ Thiết lập Bài toán:")
print(f"   • Mục tiêu: Maximize CATE × expected_value")
print(f"   • Chi phí:      channel_base_cost (NO LEAKAGE)")
print(f"   • Giá trị:     prior_spend_180d (NO LEAKAGE)")

print(f"\n✅ 5-Period Results:")
print(f"   • Avg MILP tốt hơn: ${summary['avg_gain']:,.2f} ({summary['avg_gain_pct']:+.2f}%)")
print(f"   • Greedy Speedup:       {summary['speedup']:.1f}× (Greedy faster)")

print(f"\n✅ Khuyến nghị: {summary['recommendation']}")
print(f"   • Lý do: {summary['reason']}")

print("\n🔴 CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:")
print("   • Dùng channel_base_cost thay vì spend_usd")
print("   • Dùng prior_spend_180d thay vì revenue_usd")
print("   • Phân tích tại sao MILP beats Greedy")
print("   • Simulation đa kỳ (5 periods)")

print("\n✅ Sẵn sàng cho Notebook 05 (MLOps Monitoring)")
print("=" * 70)


TÓM TẮT NOTEBOOK 04

✅ Các Phương Pháp Optimization So Sánh:
   • Greedy: Sắp xếp theo hiệu suất, chọn top
   • MILP:   Giải chính xác bài toán binary knapsack

✅ Thiết lập Bài toán:
   • Mục tiêu: Maximize CATE × expected_value
   • Chi phí:      channel_base_cost (NO LEAKAGE)
   • Giá trị:     prior_spend_180d (NO LEAKAGE)

✅ 5-Period Results:
   • Avg MILP tốt hơn: $106.30 (+1.96%)
   • Greedy Speedup:       5.6× (Greedy faster)

✅ Khuyến nghị: MILP
   • Lý do: Cost variation lớn → MILP superior

🔴 CÁC THAY ĐỔI QUAN TRỌNG SO VỚI BẢN GỐC:
   • Dùng channel_base_cost thay vì spend_usd
   • Dùng prior_spend_180d thay vì revenue_usd
   • Phân tích tại sao MILP beats Greedy
   • Simulation đa kỳ (5 periods)

✅ Sẵn sàng cho Notebook 05 (MLOps Monitoring)
